# v1 commissioning (read-only)

One section per implementation stage (S0–S9). Cells only call functions from `twobunch_deploy`; each stage saves its records under `data/` (git-ignored, copied back by hand), and the result goes in the lab-notebook commissioning page.

Nothing here writes to the machine: `tests/test_read_only.py` fails on any write call in the package or in these cells.

## S0: environment

Reads no PV. Run on the host in the ACR prod Python environment, from the repo root (Jupyter started inside tmux).

Pass:
- `import_errors` is empty;
- `code_sha256` equals the value CI printed for this PR's head commit (the `pixi run code-hash` step);
- `git_commit` is that commit;
- `epics_env` shows the host's Channel Access settings (recorded; no target value).

In [ ]:
from twobunch_deploy import session

env = session.check_env()
env

In [ ]:
session.save_record("S0", "check_env", env)

## S1: camera frames and pulse IDs

Reads one PV, `CAMR:LI20:107:RAW:ArrayData` (the DTOTR2 camera image as the driver outputs it, the same arrays the DAQ saves; `Image:ArrayData` until S2): subscribes for 30 s, then disconnects. Needs the camera on, and beam for real images. Saves the summary as JSON and, next to it (`.npz`), every frame's pulse ID and timestamps plus 10 raw frames.

Pass:
- `frames` > 0 and `latency_s.first` under about 1 s: the camera is publishing, and the frame timestamps agree with this host's clock;
- `lengths` is a single value, the camera ROI's `SizeX` × `SizeY` (278,640 = 324 × 860 for the ROI read on 10-30 at 15:26; S2 checks the geometry);
- `min` ≥ 0 and `max` ≤ 4095 (12-bit camera); `dtype` recorded;
- `pulse_id_steps` almost all 36. A 0 next to a 72, or a 35 next to a 37, is the IOC stamping a frame with a neighbouring pulse ID (8 times in 33,779 DAQ frames); any other step is a finding;
- `capture_fraction` close to 1: the callback should get every frame. Lower means frames were lost between the IOC and the notebook.

In [ ]:
from twobunch_deploy import session

summary = session.capture_test(seconds=30)
summary

## S2: frame geometry

Reads the DTOTR2 camera readbacks once (`pvs.CAMERA_SETTINGS`: ROI, orientation, exposure, gain, pixel size, the driver's reverse and output size, and where the camera array comes from and its size), then subscribes to the camera for 3 s. From each frame it cuts the deployed analysis's hand crop (`screen.HAND_CROP`, 300 × 450; at the training orientation sensor columns 396–579 and rows 270–719, padded with zeros past row 184 as the DAQ classes' crop does). The crop is taken in sensor pixels, whatever ROI the camera is set to, and oriented with the live orientation readbacks. Needs beam on DTOTR2: run the S1 section again first, with beam, then this one. Saves the summary as JSON and, next to it, the pulse IDs, the newest raw frame and every frame's hand crop (`.npz`) and the figure (`.png`).

Pass:
- no `TimeoutError`: every readback answers, so the PV names are right;
- no `ValueError`: the camera array comes from the driver (port `PS107`), isn't reversed, and is the camera ROI's size (no binning); the camera's ROI contains the hand crop; each frame is `SizeX` × `SizeY` pixels;
- `trained_shape` is [300, 450];
- `differs` lists the readbacks that aren't the training values. The four ROI readbacks may differ. Orientation, exposure (`AcquireTime_RBV`: 0.1 ms in training, 1 ms on 10-30), gain or resolution (`RESOLUTION`: 30.5 µm in training, 31.22 on 10-30) differing is a finding to understand before S3: the code follows the live orientation, but the analysis constants were set with the training values;
- the camera ROI didn't change during the 3 s: the readbacks are read once, before the capture, and a move that keeps its size wouldn't be caught. Ask the operator, or run the cell twice and compare `camera`;
- the figure's left panel shows the beam where the ops DTOTR2 display shows it, with the same handedness (compare with the display, or a screenshot of it);
- the beam lies inside the outlined hand crop.

In [ ]:
from twobunch_deploy import session

geometry = session.geometry_test(seconds=3)
geometry

## S3: background and processed frames

Reads the same PVs as S2; adds none. Each frame's hand crop goes through the deployed analysis's processing (`screen.process`, Yiheng Ye's chain with the settings of `xtcav_real_features.py`): subtract a background, replace each pixel above 1000 counts by the median of its 3 × 3 neighbourhood, then set the pixels below 35 counts to 0. Then the training's energy cut (`screen.valley`: until S4 and S5, each frame's own valley) and band statistics (`screen.band_stats`), with x scaled by the live `RESOLUTION`.

Two backgrounds, both subtracted pixel by pixel from the hand crop:
- `take_background()`: the mean of 5 s of frames with no beam on DTOTR2.
- `pedestal()`: one number, the median of every hand-crop pixel over 5 s of frames with beam on, for sessions that start with beam.

Each call saves its record under `data/<date>_S3/` (JSON, `.npz`, figure). A later session can reuse a saved background with `session.load_background(path)`, `path` being the background's `.npz` as its `path` shows it.

**Parasitic:** nothing here asks ops for anything. Take the background whenever DTOTR2 shows no beam spot (the ops display, or S2's figure), and run the rest on whatever beam the machine has. The checks that need the two-bunch beam with the TCAV on (a cut on most frames, the band statistics, `peek()`'s cut and centroids) are in the two-bunch runbook (plan §10), not here.

### Background: no beam on DTOTR2

Pass:
- `frames` about 50;
- every `frame_max` near the dark noise (the 10-30 dark frames peaked at 78–80 counts); a frame with beam reaches hundreds of counts or more. The figure's right panel is the brightest frame and must show no beam spot;
- `background.mean` near the dark level (31.3 counts on 10-30);
- `camera` recorded: the exposure, gain and ROI the background was taken at.

In [ ]:
from twobunch_deploy import session

background = session.take_background(seconds=5)
background.record

### Pedestal: beam on DTOTR2, any configuration

Pass:
- `background.mean` (the pedestal) within about 2 counts of the background's `background.mean` above. On 10-30 the beam frames gave 32–33 against 31.3: the beam's faint halo raises the median a little;
- `fraction_above_threshold` well under half (7.8% on 10-30's beam, 2.2–2.4% in the DAQ runs). Near half or more, the beam fills too much of the crop for the median to be the dark level. Near 0, there was no beam: run it again when there is.

In [ ]:
pedestal = session.pedestal(seconds=5)
pedestal.record

### Processing test: no beam on DTOTR2

Every frame of a 10 s capture, processed once with each background; the `.npz` keeps every frame's hand crop for offline checks.

Pass:
- `frames` about 100;
- `intensity_median` near 0 with both backgrounds: the threshold removes the dark noise. On the 10-30 dark frames: 0 with a live background, about 180 counts per frame with a pedestal; a beam frame sums to 5–8e5;
- `processing_s`: `median` and `max` well under 0.1 s, one frame at 10 Hz.

In [ ]:
processing_off = session.processing_test([background, pedestal], seconds=10)
processing_off

### Processing test: beam on DTOTR2, any configuration

Pass:
- `processing_s`: `median` and `max` well under 0.1 s;
- the two backgrounds' `intensity_median` agree within about 2%. On 10-30's beam, the pedestal gave 0.3–1.5% more than a live background;
- recorded, not judged here: `intensity_median` (DAQ runs 4.7–4.9e5 per frame, 10-30's beam about 8e5), `cut_found` and the band-statistics `median`s, which mean something only on the two-bunch beam.

After copying the records back, the saved crops give a frame-by-frame comparison of the two backgrounds offline (10-30: the processed beam's centroid moved by under 0.1 px, at most 0.5 px).

In [ ]:
processing_on = session.processing_test([background, pedestal], seconds=10)
processing_on

### peek(): any beam

The newest frame through the processing, with the background.

Pass:
- the figure renders: the hand crop, the processed frame and its y projection. A cut appears only where the frame shows a valley; on one bunch or with the TCAV off there may be none;
- `frames` 2 or more and `age_s` under about 1 s: the frame is current;
- `processing_s` recorded.

In [ ]:
session.peek(background)

## S4: TCAV readbacks and tags

Adds two PVs: the XTCAV's amplitude and phase readbacks (`pvs.TCAV_AMPLITUDE`, MV, and `pvs.TCAV_PHASE`, deg; the DAQ's names, checked by hand on srv01 on 11-02: 30 Hz, timestamped with the pulse). Each capture subscribes to both alongside the camera (`acquire.capture`), and each frame takes the readbacks with its pulse ID. Each frame is tagged as Yiheng Ye's `classify_tcav_shots` does (the deployed analysis):
- `+` and `-`: amplitude within 0.25 MV of the mean of the amplitudes above 10 MV, and phase within 1° of +90 and −90;
- `off`: amplitude more than 0.5 MV below the mean amplitude of the `+` and `-` frames (with the TCAV off throughout, below 9.5 MV);
- `neither`: any other readback;
- `unmatched`: no readback with the frame's pulse ID.

The readbacks don't show the toggler's off windows as off: with the klystron on standby, they measure the standby pulse (amplitude near 22 MV, phase slewing to the next polarity) while the beam is unstreaked. Those frames are mostly `neither`; `off` catches the amplitude dip at the trigger switch, a few frames per off window. `step()` records each frame's beam x RMS, which shows whether its off frames were unstreaked.

The `+` and `-` frames less than 1 s after their polarity window began are dropped: the screen keeps settling for about 1 s after the phase locks. A window begins when the polarity changes, after an off frame, or after 5 or more `neither` frames. Records go under `data/<date>_S4/`.

### tcav_test(): any beam, whatever state the TCAV is in

10 s of frames with the readbacks; the `.npz` keeps every frame's pulse ID, readbacks and tag, and every update.

Pass:
- no `TimeoutError`: both PVs answer;
- `updates`: each PV's `count` about three per frame, its `pulse_id_steps` 12 (30 Hz, as read by hand on 11-02). Far fewer updates mean frames go unmatched, and the matching needs another design before a TCAV window;
- `matched_fraction` near 1;
- `tags` as the TCAV is: off → all `off` (amplitude near 0; 11-02 by hand: 0.015–0.02 MV); on at one polarity → `+` or `-` (`range`: amplitude near 22 MV, phase within 1° of ±90); toggling → all four tags;
- `range` in the units expected: MV and degrees.

In a TCAV window (else at the start of two-bunch shift 1): the tags separate off, + and −; frames just after each flip whose image disagrees with its tag set `drop_s`.

In [ ]:
tcav = session.tcav_test(seconds=10)
tcav

### peek(): the newest frame's tag

Pass: the title shows the newest frame's TCAV tag, amplitude and phase, as the TCAV is. The cut is still the frame's own valley, without row correction (`step()` has both).

In [ ]:
session.peek(background)

## S5: row correction and step readings

Adds no PV. `step()` captures for `seconds` with the TCAV readbacks, then reads the frames as the deployed analysis reads a DAQ step (`screen.read_step`; the v7 replay):
1. each `+`, `-` and `off` frame through the S3 processing, with the background;
2. the row-correction reference from the step's off frames: their mean, a Gaussian fitted in x to each row with signal, and each row's shift to the crop's centre; each `+` and `-` frame's rows shifted by it;
3. the step's energy cut: the median of the kept `+` frames' own valleys; each `+` and `-` frame's band statistics at it;
4. the reading from the kept frames: spacing (half-difference of the drive–witness offsets over the streak), drive σz, drive charge fraction, edge fraction, cut sensitivity.

The reading is valid only if every check passes: at least 15 kept frames with statistics per polarity, an off frame (no reference otherwise), a cut, |spacing| above 60 µm, an edge fraction below 0.01, and beam (each polarity's kept frames sum to a median of at least 1e5 counts; beam frames sum to 5–9e5). Everything the frames allow is computed and recorded either way.

`streak_um_per_deg` is the TCAV calibration in µm on the screen per degree, at the live pixel size (`RESOLUTION`, 31.22 µm on 10-30): a calibration measured at another pixel size must be rescaled first (the DAQ runs' 1050–1113 µm/deg are at 30.5). On parasitic beam any value does: 1136 is the DAQ's 1110 rescaled to 31.22.

Records go under `data/<date>_S5/`: the reading, its checks, the counts per tag, the reference fit, timing, the session's peak memory, and each frame's pulse ID, readbacks, tag, drop and band statistics (JSON); every frame's hand crop with the same, the reference and the two median frames (`.npz`). `r.show()` draws the step and saves the figure next to them.

### step(): any beam

Pass:
- it returns. With the TCAV off, `valid` is False and `checks` say why: no `+` or `-` frames, no cut, no beam check;
- with beam and the TCAV off, every frame is `off` and the reference is built from all of them: `reference.rows_fitted` above 0, `shift_columns` and `off_x_rms_mm` recorded (unstreaked: about 0.2 mm);
- `timing_s`: `capture` about `seconds`; `reading` well under the step period (1.5–2 s for about 70 frames on the DAQ steps);
- `peak_memory_mb` recorded (frames at full sensor are 2.5 MB each);
- `r.show()` renders: the readback panels show each frame's tag.

In a TCAV window (else at the start of two-bunch shift 1): steps with both polarities and at least 15 kept frames each, the reference's stability from step to step, the timing cycle. Two-bunch runbook: 20–30 steps with the machine left alone.

In [ ]:
r = session.step(background, streak_um_per_deg=1136, seconds=15)
{k: r.record[k] for k in ("valid", "checks", "tags", "reference", "cut_row", "reading", "timing_s", "peak_memory_mb")}

In [ ]:
r.show()

## S6, S7: L1 and L2 phase PVs

Adds 16 PVs, checked by hand on srv01 (`to_srv01/phase_pv_check.py` on 11-04, `to_srv01/l1_readback_probe.py` on 11-05). L1 (`pvs.L1`), the LI11 klystrons 11 and 21: the phase and amplitude readbacks `FB:FAST_PACTBR` and `FB:FAST_AACTBR` (the DAQ's `FB:FAST_PACT` and `FB:FAST_AACT`, BSA PVs read at beam rate: Zack, 11-05), the phase setpoints `PREQ` (in no DAQ; a feedback writes both) and the subbooster phase setpoints `SSSB_PDES`. L2 (`pvs.L2`): the subbooster phase setpoints `PDES` of sectors 11–14 and their readbacks `PHAS` (the DAQ's nonBSA names). The L2 multiknob moves LI12–14's 1:1; LI11's phases L2's klystrons 4–8 (F2_live_model), and the multiknob doesn't move it. All read over Channel Access.

All 16 are monitored with the TCAV readbacks during every capture (`step()`, `tcav_test()`, `peek()`), so those calls stop with a `TimeoutError` naming any of them that doesn't answer. `step()` records each PV's number of updates, median, first and last value and exact zeros (`acquire.summarise`) under `phases`; its `.npz` keeps every update with its pulse ID. Nothing is checked at run time: a setpoint that changed during a step shows as first ≠ last.

### phases_test(): any beam, or none

60 s of updates of the 16 PVs, without the camera. The `.npz` keeps every update; the figure shows each PV against time. Only the L1 readbacks carry the shot's pulse ID; the other PVs carry the IOC's or gateway's time, so their `pulse_id_steps` mean nothing.

Pass (the hand-check values in brackets: L1 readbacks and `PREQ` 11-05 at 5 Hz beam rate, the rest 11-04):
- no `TimeoutError`: all 16 answer;
- each `FAST_PACTBR` and `FAST_AACTBR`: one update per shot, so `pulse_id_steps` all 360 / beam rate (72 at 5 Hz), no exact zeros with beam; `FAST_PACTBR` within about 0.1° of `PREQ` in median (+0.02° and +0.015°), per-shot std about 0.15–0.2° (0.16°, 0.15°), 11 and 21 within about 0.05° of each other in median; `FAST_AACTBR` steady (0.500 and 0.5725, std below 0.001);
- the two `PREQ`s equal, changing at 5 Hz while their feedback runs, whatever the beam rate (11-05: 301 updates in 60 s with 10 Hz beam, std 0.08°; on 11-04 −24.95 to −24.53° over 60 s); one update if the feedback is off;
- `SSSB_PDES` −18.03, one update;
- the LI12–14 `PDES` moving together, as the multiknob display shows them (about 1 update a second, the same moves in all three, std 0.05°; offsets between sectors of 0.85° and 2.2°); LI11's 0, one update;
- each `PHAS` within about 2° of its `PDES` (an update every 5 s or so; medians 0.4°, 1.6° and 0.7° from LI12, 13 and 14's `PDES`). −10000 is an invalid-reading sentinel (once in 60 s): the median ignores it, but `first`, `last`, `min` and `std` don't.

If ops move L1 or L2 during the session: `PREQ` moves and `FAST_PACTBR` follows, and the LI12–14 `PDES` move together.

In [ ]:
phases = session.phases_test(seconds=60)
phases

### step(): the L1 and L2 PVs over a step

Uses `background` from S3. Pass:
- `phases` holds all 16 PVs; each `FAST_PACTBR` has about one update per frame, each `PREQ` about 5 a second;
- `r.show()` renders: the fourth panel shows both `FAST_PACTBR`s per shot with their medians and both `PREQ`s as steps (the readbacks scatter around the setpoint), and its title the `SSSB_PDES` and the `FAST_AACTBR` medians; the bottom panel shows the LI12–14 subbooster `PDES` during the step, each minus its median (they should move together), and its title LI11's `PDES` and the four `PHAS`.

In [ ]:
r = session.step(background, streak_um_per_deg=1136, seconds=15)
r.record["phases"]

In [ ]:
r.show()

## S8: bunch length, drift monitors and magnets

Adds 41 PVs. Check their names by hand on srv01 first (`to_srv01/s8_pv_check.py`): the bunch-length names are a guess from the DAQ's (`BLEN:...:BRAW` in its BSA lists, so probably `BRAWBR` live, as for the L1 readbacks).
- **Bunch length** (`pvs.BLEN`): the raw signals of the injector, BC11, BC14 and sector 20 monitors, read at beam rate. Absolute counts aren't comparable across sessions, only changes within one.
- **Drift** (`pvs.DRIFT`): F2_live_model's L2 and L3 energy gain, chirp and fudge and its heartbeat; the LI14 klystron pair 4/5's phase and amplitude setpoints and readbacks.
- **Magnets** (`pvs.MAGNETS`): the supply readbacks of the final-focus and spectrometer quads (named as in F2_live_model's map), the spectrometer dipole B5D36, the four final-focus steering correctors, the two LI19 quads that launch into BC20, and the six BC20 chicane sextupoles.

The bunch-length and drift PVs are monitored with the others during every capture (`step()`, `tcav_test()`, `peek()`), so those calls stop with a `TimeoutError` naming any that doesn't answer; `step()` records each one's summary under `drift`. The magnets are read once at the start of each step (`magnets`: None for one that doesn't answer) and by `optics_test()`.

### drift_test(): any beam, or none

60 s of updates of the bunch-length and drift PVs, without the camera. The `.npz` keeps every update; the figure shows each PV against time.

Pass (the hand check's values in brackets once it has run):
- no `TimeoutError`: all 19 answer;
- each bunch-length signal: one update per shot with beam (`pulse_id_steps` all 360 / beam rate), timestamped with the shot's pulse ID; exact zeros only without beam;
- F2_live_model: L2's values change about once a second, the heartbeat counts up about once a second (11-04: 3580 MeV, −3669 MeV, fudge 1.163; L3 5265 MeV, −8.8 MeV, 1.045, unchanged in a minute);
- LI14 klystrons 4 and 5: `PDES` near −63° and +66° and moving about 1° (11-04); `PHAS` near `PDES`; `ADES` and `AMPL` steady.

In [ ]:
drift = session.drift_test(seconds=60)
drift

### optics_test(): any time

One read of each magnet supply against TEST_15668's (the three DAQ runs agree within 0.01%).

Pass: all 22 answer; record each `difference` (the correctors sit near 0, so their percent is large) and `difference_pct`. Within ±1% is the trained optics (the before-shift checklist); a larger difference is a fact about the session, not a failure of this stage.

In [ ]:
optics = session.optics_test()
optics

### step(): the S8 PVs over a step

Uses `background` from S3. Pass:
- `drift` holds all 19 monitored PVs, each bunch-length signal with about one update per frame; `magnets` all 22 values;
- `r.show()` renders: the bottom panel shows the BC11 and BC14 signals per shot as a change from their medians in percent, and its title the injector's and sector 20's medians and each drift PV, first → last.

In [ ]:
r = session.step(background, streak_um_per_deg=1136, seconds=15)
r.record["drift"], r.record["magnets"]

In [ ]:
r.show()

## S9: the v1 console

Adds no PV. Run `console.ipynb` top to bottom on whatever beam: its checklists, then every call. New here: `check_pvs()`, `status()`, `watch()` and `peek_strip()`.

Pass:
- every call runs and saves its record under `data/<date>_<stage>/`;
- `check_pvs()`: all 77 answer (`missing` empty); ages make sense (setpoints can be days old);
- `watch()` replaces its figure about once a second and stops by itself; `peek_strip()` renders, with a spacing only if the capture has both polarities;
- `status()` counts the steps taken and shows the last one;
- after a kernel restart, `status()` still shows them.

## W2 (v2): writes with the beam stopped at TD11

The first writes to the machine, with arbitrary moves and no policy (v2 plan, 11-06). Everything is written through `actuate.py`: both L1 phase setpoints (`PREQ`, the same delta to each) and the L2 phase multiknob, each write at most one action (0.25° L1, 0.3° L2), the net move within ±1.0° L1 and ±1.5° L2. Every write is logged in the session's ledger (`data/<date>_V2/ledger_*.jsonl`) before it is sent and after its readback check: both `PREQ`s must read the new value; the LI12–14 subbooster setpoints must move by the delta.

**Before:** the beam stopped at TD11 (L1 and BC11 still see beam, L2 none); ops present; the energy feedbacks on; the L2 bunch-length feedback off; the `PREQ` feedback still **on** for the first cell; starting values noted from ops' displays.

In [ ]:
from twobunch_deploy import actuate, session

actuate.pvaccess_version()

### arm() refused while the PREQ feedback is on

Pass: `RuntimeError: PREQ updated ... its feedback is on`, and no lock left (`actuate.state()`). Then ops turn the `PREQ` feedback off.

In [ ]:
actuate.arm("Ryan")

In [ ]:
actuate.state()

### Hold, 10 min with PREQ frozen

With the `PREQ` feedback off and before arming: nothing is written. Gives the L1 readbacks' step-mean scatter with no feedback motion (trained 0.02°) and the BC11 bunch-length drift with its loop off.

In [ ]:
hold = session.hold_test(seconds=600)
hold

### Arm, round trips

Pass: `arm()` returns the starting setpoints. Each move returns a ledger entry with `state` `done`; `PREQ` and `FAST_PACTBR` (ops' displays, and `hold_test` below) follow L1; the subbooster setpoints move 1:1 with L2. After the four moves, `state()` shows a net 0 and nothing pending.

In [ ]:
actuate.arm("Ryan")

In [ ]:
actuate.move_l1(0.1)

In [ ]:
actuate.move_l1(-0.1)

In [ ]:
actuate.move_l2(0.1)

In [ ]:
actuate.move_l2(-0.1)

In [ ]:
actuate.state()

### Staircases

Pass:
- each move `done`; the move past the budget refused (`refused` names the budget);
- L1: in each move's `hold_after` against `hold_before`, `FAST_PACTBR` moves by the step (over the 10 s holds) and the BC11 bunch-length signal by about 5–10% per 0.25°: its sign against the simulator's L1;
- L2: the LI12–14 subbooster setpoints move by the step;
- back at the starting values at the end.

In [ ]:
l1 = session.staircase("L1", 0.25, 4)
l1["refused"]

In [ ]:
l2 = session.staircase("L2", 0.3, 5)
l2["refused"]

### End

`undo_all()` reverses this session's writes (here, back through the path already walked), then disarm. Pass: the setpoints equal the values noted before; ops restore the `PREQ` feedback (resetting its setpoint first) and the L2 bunch-length feedback as agreed.

In [ ]:
actuate.undo_all()

In [ ]:
actuate.state()

In [ ]:
actuate.disarm()